In [1]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt

import os

In [2]:
path = os.getcwd()
print(path)

/Users/eleonoracappuccio/xai-visualization_rules_fi/notebooks


# Loading a dataset and preparing it

In [3]:
datasets=['titanic_c.csv','german_credit.csv']

In [4]:
source_file = f'../datasets/{datasets[1]}'
class_field = 'default'
# Load and transform dataset
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

In [5]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

## Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [6]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])


Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [7]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)

Select a new instance to be classfied by the model and print the predicted class.

In [8]:
inst = X_train.iloc[128].values
print('Instance ',inst)
print('True class ',Y_train.iloc[128])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [  27 4526    4    2   32    2    2    0    0    0    1    0    1    0
    0    0    0    0    0    0    0    0    0    1    0    0    1    0
    0    0    0    0    1    0    0    0    0    0    0    1    0    0
    1    0    0    1    0    0    0    1    0    1    0    0    0    0
    1    0    1    0    1]
True class  0
Predicted class  [0]


In [9]:
real_inst = inst
real_inst

array([  27, 4526,    4,    2,   32,    2,    2,    0,    0,    0,    1,
          0,    1,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    1,    0,    0,    1,    0,    0,    0,    0,    0,    1,
          0,    0,    0,    0,    0,    0,    1,    0,    0,    1,    0,
          0,    1,    0,    0,    0,    1,    0,    1,    0,    0,    0,
          0,    1,    0,    1,    0,    1])

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [10]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:].values}
explainer.fit(config)

In [11]:
exp = explainer.explain(inst)

In [12]:
exp.exp

[array([-9.36117846e-03,  1.12032620e-02, -9.27695620e-03,  2.34854277e-02,
         7.16980504e-03, -6.74858143e-03,  1.62828062e-03,  8.94296419e-03,
         1.05535082e-02,  1.18873255e-03,  5.97444048e-02,  6.01048718e-03,
         4.42259296e-02,  8.23171215e-04, -5.29116292e-05,  3.35190747e-03,
         5.34567691e-03,  1.59816407e-03,  1.32973681e-02, -3.37325406e-04,
        -6.45782153e-03, -1.92963142e-04, -3.43301732e-05, -1.02040224e-02,
        -9.14099568e-05, -1.88173738e-05,  3.73942711e-02,  1.67373617e-02,
        -2.71268261e-04,  4.33195433e-03, -6.42038496e-03,  4.19965582e-03,
        -6.04810835e-02,  2.52984889e-03, -4.70397752e-03,  1.08855784e-03,
         3.60347919e-03,  2.57403676e-03,  2.60612959e-03,  6.01817322e-03,
         1.27007955e-03, -1.29012630e-04, -5.39136185e-04,  2.42969702e-03,
         7.47511264e-03,  2.09428070e-02,  7.15806453e-03,  1.95509790e-02,
        -2.60691783e-02, -6.28495831e-03,  5.63396486e-05,  5.83210604e-03,
         3.4

In [13]:
shap_feature_importance=exp.exp

In [14]:
exp.plot_features_importance()

alt.VConcatChart(...)

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.

In [15]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [16]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


In [17]:
X_scaled

array([[-0.7335121 , -0.71300074,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766, -0.61086948,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766,  0.3606869 , -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415],
       ...,
       [-0.23159766, -0.25658997,  0.0547138 , ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.23159766,  1.97159246, -1.72666575, ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.48255488, -0.06429887, -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415]])

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.

## LIME tabular explainer

In [18]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())# è una lista di tuple

[('other_debtors=co-applicant', -1.9989975283832505e-09), ('credit_history=all credits at this bank paid back duly', -1.619057256952488e-09), ('present_emp_since=unemployed', -1.4905987198621018e-09), ('other_debtors=none', 1.2131142878338074e-09), ('housing=for free', -7.443228548969372e-10), ('job=management/ self-employed/ highly qualified employee/ officer', -5.874146449430365e-10), ('property=unknown / no property', -5.230779938474698e-10), ('housing=own', 4.5818578941716493e-10), ('savings=unknown/ no savings account', -4.278280175476622e-10), ('people_under_maintenance', 3.8251280233989043e-10), ('job=skilled employee / official', 3.576130789053601e-10), ('telephone=none', 3.3598960612581333e-10), ('foreign_worker=yes', 3.2955335465469115e-10), ('credit_amount', 3.272173984548671e-10), ('credits_this_bank', 3.259641616846247e-10), ('purpose=car (new)', -3.0972974140258673e-10), ('savings=... < 100 DM', 3.030992665799685e-10), ('credit_history=existing credits paid back duly till

In [19]:
lime_feature_importance=lime_exp.exp.as_list()
lime_feature_importance

[('other_debtors=co-applicant', -1.9989975283832505e-09),
 ('credit_history=all credits at this bank paid back duly',
  -1.619057256952488e-09),
 ('present_emp_since=unemployed', -1.4905987198621018e-09),
 ('other_debtors=none', 1.2131142878338074e-09),
 ('housing=for free', -7.443228548969372e-10),
 ('job=management/ self-employed/ highly qualified employee/ officer',
  -5.874146449430365e-10),
 ('property=unknown / no property', -5.230779938474698e-10),
 ('housing=own', 4.5818578941716493e-10),
 ('savings=unknown/ no savings account', -4.278280175476622e-10),
 ('people_under_maintenance', 3.8251280233989043e-10),
 ('job=skilled employee / official', 3.576130789053601e-10),
 ('telephone=none', 3.3598960612581333e-10),
 ('foreign_worker=yes', 3.2955335465469115e-10),
 ('credit_amount', 3.272173984548671e-10),
 ('credits_this_bank', 3.259641616846247e-10),
 ('purpose=car (new)', -3.0972974140258673e-10),
 ('savings=... < 100 DM', 3.030992665799685e-10),
 ('credit_history=existing credit

In [20]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer

In [21]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [22]:
exp.plotRules()

In [23]:
exp.plotCounterfactualRules()

In [24]:
rules =exp.expDict['rule']['premise']

In [25]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [26]:
for r in rules:
    print(r['att'])

age
credit_amount
purpose=retraining
duration_in_month
purpose=furniture/equipment
foreign_worker=no
purpose=domestic appliances
savings=.. >= 1000 DM 
purpose=(vacation - does not exist?)
credit_history=critical account/ other credits existing (not at this bank)
people_under_maintenance


In [27]:
df_rules = pd.DataFrame.from_records(rules)

In [28]:
df.describe()

,duration_in_month,credit_amount,installment_as_income_perc,present_res_since,age,credits_this_bank,people_under_maintenance,account_check_status=0 <= ... < 200 DM,account_check_status=< 0 DM,account_check_status=>= 200 DM / salary assignments for at least 1 year,...,housing=rent,job=management/ self-employed/ highly qualified employee/ officer,job=skilled employee / official,job=unemployed/ unskilled - non-resident,job=unskilled - resident,telephone=none,"telephone=yes, registered under the customers name",foreign_worker=no,foreign_worker=yes,default
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,0.269000,0.274000,0.063000,...,0.179000,0.148000,0.630000,0.022000,0.2000,0.596000,0.404000,0.037000,0.963000,0.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.443662,0.446232,0.243085,...,0.383544,0.355278,0.483046,0.146757,0.4002,0.490943,0.490943,0.188856,0.188856,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,1.000000,0.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.0000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,1.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.0000,1.000000,1.000000,0.000000,1.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,1.000000,1.000000,1.000000


In [29]:
df_range=pd.concat(
    {'min':X_train.min(),
     'max':X_train.max(),
     'std':X_train.std(),
     'q1':X_train.quantile(0.25),
     'median':X_train.quantile(0.50),
     'q3':X_train.quantile(0.75),
     },axis=1)

In [30]:
df_range=df_range.reset_index()

In [31]:
df_range

,index,min,max,std,q1,median,q3
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00
3,present_res_since,1,4,1.097089,2.00,3.0,4.00
4,age,19,75,10.954080,27.00,33.0,41.25
...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00
57,telephone=none,0,1,0.491104,0.00,1.0,1.00
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00


In [32]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,std,q1,median,q3,op,thr,is_continuous
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00,>,-1.940701,True
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50,>,-439.644349,True
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00,NaN,NaN,NaN
3,present_res_since,1,4,1.097089,2.00,3.0,4.00,NaN,NaN,NaN
4,age,19,75,10.954080,27.00,33.0,41.25,<=,20.726173,True
...,...,...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00,NaN,NaN,NaN
57,telephone=none,0,1,0.491104,0.00,1.0,1.00,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00,NaN,NaN,NaN
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00,<=,0.716841,True


In [33]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if row['op']== '>' or row['op']== '>=':
        thr2_list.append(row['max'])
        continue
    if row['op']== '<' or row['op']== '<=':
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

,index,min,max,std,q1,median,q3,op,thr,is_continuous,thr2
0,duration_in_month,4,60,11.962777,12.00,18.0,24.00,>,-1.940701,True,60.0
1,credit_amount,338,15945,2674.942042,1360.75,2319.5,3974.50,>,-439.644349,True,15945.0
2,installment_as_income_perc,1,4,1.123528,2.00,3.0,4.00,NaN,NaN,NaN,NaN
3,present_res_since,1,4,1.097089,2.00,3.0,4.00,NaN,NaN,NaN,NaN
4,age,19,75,10.954080,27.00,33.0,41.25,<=,20.726173,True,19.0
...,...,...,...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,0.400286,0.00,0.0,0.00,NaN,NaN,NaN,NaN
57,telephone=none,0,1,0.491104,0.00,1.0,1.00,NaN,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,0.491104,0.00,0.0,1.00,NaN,NaN,NaN,NaN
59,foreign_worker=no,0,1,0.196099,0.00,0.0,0.00,<=,0.716841,True,0.0


In [34]:
features=df_viz['index'].to_list()
features

['duration_in_month',
 'credit_amount',
 'installment_as_income_perc',
 'present_res_since',
 'age',
 'credits_this_bank',
 'people_under_maintenance',
 'account_check_status=0 <= ... < 200 DM',
 'account_check_status=< 0 DM',
 'account_check_status=>= 200 DM / salary assignments for at least 1 year',
 'account_check_status=no checking account',
 'credit_history=all credits at this bank paid back duly',
 'credit_history=critical account/ other credits existing (not at this bank)',
 'credit_history=delay in paying off in the past',
 'credit_history=existing credits paid back duly till now',
 'credit_history=no credits taken/ all credits paid back duly',
 'purpose=(vacation - does not exist?)',
 'purpose=business',
 'purpose=car (new)',
 'purpose=car (used)',
 'purpose=domestic appliances',
 'purpose=education',
 'purpose=furniture/equipment',
 'purpose=radio/television',
 'purpose=repairs',
 'purpose=retraining',
 'savings=.. >= 1000 DM ',
 'savings=... < 100 DM',
 'savings=100 <= ... <

In [35]:
def data_to_plot(
        feature_names=feature_names, real_feature_names=real_feature_names,
        instance_number=None, x_train=None, rules=None,
        feature_importance=None):
    feature_list =[]
    #Convert the list of tuples generated by lime in a dict
    if feature_importance is 'lime':
        lime_dict = {}
        for (key, value) in lime_feature_importance:
            lime_dict.setdefault(key, value)
    for i, el in enumerate(feature_names):
        f ={}
        if el in numeric_columns:
            f['type'] = 'numeric'
            f['name'] = el
            f['rname'] = real_feature_names[i]
            if x_train is not None:
                f['min'] = x_train[el].min()
                f['max'] = x_train[el].max()
                f['q1'] = x_train[el].quantile(0.25)
                f['median'] = x_train[el].quantile(0.25)
                f['q3'] = x_train[el].quantile(0.75)
                f['mean'] = x_train[el].mean()
                f['std'] = x_train[el].std()
        else:
            f['type'] = 'categorical'
            f['name'] = el
            f['rname'] = el.split('=')[0]
            f['category'] = el.split('=',1)[1]
            if x_train is not None:
                f['count'] = x_train[el].sum()

        if feature_importance is 'lime':
            f['feature_importance'] = lime_dict[el]
        if feature_importance is 'shap':
            f['feature_importance'] = shap_feature_importance[1][i]
        feature_list.append(f)
    df =pd.DataFrame.from_records(feature_list)

    if instance_number:
        inst = X_train.iloc[instance_number].values
        df['inst'] = inst
    if rules is not None:
        df_rules = pd.DataFrame.from_records(rules)
        df = df.merge(df_rules,how='left',left_on='name',right_on='att')
        df = df.drop('att', axis=1)
        thr2_list=[]
        for i, row in df.iterrows():
            if row['op']== '>' or row['op']== '>=':
                thr2_list.append(row['max'])
                continue
            if row['op']== '<' or row['op']== '<=':
                thr2_list.append(row['min'])
                continue
            else:
                thr2_list.append(np.nan)
        df['thr2'] = thr2_list
    df.sort_values(by=['feature_importance'], key=lambda x: abs(x), ascending=False, inplace=True)
    return df

In [36]:
def single_rule_plot_n(dataframe,rw):
    data = dataframe[dataframe['name'] == rw['name']]
    p=alt.Chart(
        data
    ).mark_point(
        color='black' if rw['is_continuous'] == True else 'black',
        size=20,
        shape='diamond'
    ).encode(
        x=alt.X(
            field='inst',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        tooltip=[alt.Tooltip(field='inst', title=rw['name'])]
    )

    t_min = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=-10,
        align='right'
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None
        ),
        text='min:N'
    )

    t_max = alt.Chart(
        data
    ).mark_text(
        color='black',
        dx=10,
        align='left'
    ).encode(
        x=alt.X(
            field='max',
            type='quantitative',
            title=None
        ),
        text='max:N'
    )

    q1_m = alt.Chart(
        data
    ).mark_bar(
        color='#DAE7E8',
        size=12
    ).encode(
        x=alt.X(
            field='q1',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='median'
        ),
    )

    m_q3 = alt.Chart(
        data
    ).mark_bar(
        color='#A8B9BF',
        size=12
    ).encode(
        x=alt.X(
            field='median',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2 = alt.X2(
            field='q3'
        ),
    )

    b =alt.Chart(
        data
    ).mark_bar(
        color='#f28e46',size=5
    ).encode(
        x=alt.X(
            field='thr',
            type='quantitative',
            title=None,
        ),
        x2='thr2',
        y=alt.Y(
            field='index',
            type='nominal',
            title=None
        ),

    )



    l =alt.Chart(
        data
    ).mark_bar(
        color='grey',size=1
    ).encode(
        x=alt.X(
            field='min',
            type='quantitative',
            title=None,
            scale= alt.Scale(domain=(rw['min'], rw['max']), clamp=True, nice=False)
        ),
        x2='max',
        y=alt.Y(field='index',type='nominal',title=None, axis=alt.Axis(labels= False, ticks=False))
    )


    return alt.layer(l,q1_m,m_q3,b,t_min,t_max,p).properties(
        height=12,
        width=399,
    )

In [37]:
def single_feature_importance_plot(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).mark_bar(
    ).encode(
        x=alt.X(
            field='feature_importance',
            type='quantitative',
            title=None,
            scale=alt.Scale(
                domain=(dataframe['feature_importance'].min(), dataframe['feature_importance'].max()),
                nice=False
            )
        ),
        y=alt.Y(
            field='index',
            type='nominal',
            title=None,
            axis=None
        ),
        color=alt.condition('datum.feature_importance > 0', alt.value('#2C0AD1'), alt.value('#DB2C8F')),
        tooltip=[alt.Tooltip(field="name"),alt.Tooltip(field="feature_importance")]
    )
    return chart.properties(
        height=12,
        width=50
    )

In [38]:
def single_instance_text(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).mark_text(
        color='black',
        align='left',
        dx=-30,
        fontWeight='bold'
    ).encode(
            text=alt.Text(
            field='inst',
            type='quantitative',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=10
    )

In [39]:
def single_index_text(dataframe, rw):
    data = dataframe[dataframe['name'] == rw['name']]
    chart = alt.Chart(
        data
    ).transform_calculate(
        label ="datum.type=='categorical' ? datum.name : datum.name +' = '+ datum.inst" #  datum.index +' = '+ datum.inst
    ).mark_text(
        color='black',
        align='left',
        dx=-100,
    ).encode(
            text=alt.Text(
            field='label',
            type='nominal',
            title=None
        )
    )
    return chart.properties(
        height=12,
        width=50
    )

In [40]:
def single_rule_plot_q(dataframe, rw):
    name= rw['name'].split('=')[0]
    data = dataframe[dataframe['rname'] == name]
    b = alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.inst==1',alt.value('darkgrey'),alt.value('lightgrey'))
    )

    r=alt.Chart(
        data
    ).mark_bar(
        stroke='white'
    ).encode(
        x=alt.X(
            field='count',
            type='quantitative',
            title=None,
            # stack="normalize"
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    
    dot =alt.Chart(
        data
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'descending')]
    ).mark_point(
        size=40,
        shape='diamond',
        color='black'
    ).encode(
        x=alt.X(
            field='count_start',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        x2=alt.X2(
            field='count_end',
            type='quantitative',
            title=None,
            # stack="normalize",
        ),
        y=alt.Y(
            field='rname',
            type='nominal',
            axis=None
        ),
        detail='index:N',
        # color=alt.condition('datum.is_continuous',alt.value('#f28e46'),alt.value('white')),
        # opacity=alt.condition('datum.is_continuous',alt.value(1),alt.value(0.001)),
        #tooltip=[alt.Tooltip(field='category', title=name), alt.Tooltip(field='count')]
    )
    

    return alt.layer(b,r).properties(
        height=10,
        width=400,
    )

In [41]:
def plot_rules(dataframe, only_rules=False):
    tx_list=[]
    ti_list=[]
    rp_list=[]
    fi_list=[]
    if only_rules==True:
        dataframe = dataframe[dataframe['is_continuous']==True]
    for i, row in dataframe.iterrows():
        if row['inst']!=0: # or 
            stx = single_instance_text(dataframe, row)
            sti = single_index_text(dataframe, row)
            if row['type']== 'numeric':
                srp = single_rule_plot_n(dataframe, row)
            else:
                srp = single_rule_plot_q(dataframe, row)
            sfi = single_feature_importance_plot(dataframe, row)
            tx_list.append(stx)
            ti_list.append(sti)
            rp_list.append(srp)
            fi_list.append(sfi)
    tx_concat=alt.vconcat(*tx_list)
    ti_concat=alt.vconcat(*ti_list)
    rp_concat=alt.vconcat(*rp_list, title='Rule')
    fi_concat=alt.vconcat(*fi_list, title='FI')
    final_chart = alt.hconcat(fi_concat, rp_concat, ti_concat)

    return final_chart.configure_concat(
        spacing=3
    ).configure_axis(
        grid=False
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        domain=False,
        ticks=False
    ).configure_title(
        fontWeight='bold',
        anchor='middle',

    )

# Function to prepare data for plotting

In [42]:
df_v = data_to_plot(feature_names=feature_names, real_feature_names=real_feature_names, instance_number=3, x_train=X_train, rules=rules, feature_importance='shap')
df_v

,type,name,rname,min,max,q1,median,q3,mean,std,feature_importance,category,count,inst,op,thr,is_continuous,thr2
32,categorical,present_emp_since=... < 1 year,present_emp_since,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.060481,... < 1 year,120.0,0,NaN,NaN,NaN,NaN
10,categorical,account_check_status=no checking account,account_check_status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.059744,no checking account,276.0,0,NaN,NaN,NaN,NaN
12,categorical,credit_history=critical account/ other credits...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.044226,critical account/ other credits existing (not ...,205.0,0,<=,0.908596,True,NaN
26,categorical,savings=.. >= 1000 DM,savings,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.037394,.. >= 1000 DM,37.0,0,<=,0.717686,True,NaN
48,categorical,other_installment_plans=none,other_installment_plans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.026069,none,563.0,1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24,categorical,purpose=repairs,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000091,repairs,10.0,0,NaN,NaN,NaN,NaN
50,categorical,housing=for free,housing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000056,for free,75.0,0,NaN,NaN,NaN,NaN
14,categorical,credit_history=existing credits paid back duly...,credit_history,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000053,existing credits paid back duly till now,362.0,1,NaN,NaN,NaN,NaN
22,categorical,purpose=furniture/equipment,purpose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000034,furniture/equipment,6.0,0,<=,0.183708,True,NaN


In [43]:
plot_rules(df_v, only_rules=False)

alt.HConcatChart(...)

In [44]:
#qui da per scontato che ci sia per forza almeno un valore categorico e uno numerico nel dataset e se non è così si schianta
def dot_on_categories(dataframe,sort_by_rules=True):
    base =alt.Chart(
        dataframe
    ).transform_stack(
        stack='count',
        as_=['count_start','count_end'],
        groupby=['rname'],
        sort=[alt.SortField('count', 'ascending')]
    ).transform_calculate(
        midStack='(datum.count_start+datum.count_end)/2'
    )
    
    bar = base.mark_bar(stroke='white').encode(
        x='count_start:Q',
        x2='count_end:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='name:N',
        color=alt.condition('datum.is_continuous',alt.value("#f4dd4d"),alt.value('lightgrey')),
        tooltip=[alt.Tooltip('category')]
    )
    
    dot= base.mark_point(
        color='black',
        shape='diamond',
        fill='black',
        size=30
    ).encode(
        x='midStack:Q',
        y=alt.Y('rname:N',sort=["is_continuous", "feature_importance"]),
        detail='name:N',
        opacity=alt.condition('datum.inst==1',alt.value(0.6),alt.value(0))
    )

    return (bar+dot).properties(
        width=200,
    ).configure_axisX(
        disable=True
    ).configure_axisY(
        labelPadding=10,
        labelFontSize=12,
        domain=False,
        ticks=False,
        title=None
    ).configure_axis(
        grid=True
    ).configure_view(
        strokeWidth=0,
        stroke='lightgray'
    )
dot_on_categories(df_v)

alt.LayerChart(...)